In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager(project_id="f5b4fc7d-978f-45be-9530-b38db8ef5046")

In [ ]:
# Prepare all of our nodes!
# This takes a while...
slice_name = "rucompart-test"
image = 'default_ubuntu_20'
slice = fablib.new_slice(name = slice_name)
node_fmt = "node-{n}"
node_count = 16
nodes = []
for n in range(node_count):
    nodes.append(slice.add_node(name = node_fmt.format(n=n), image = image, cores = 2, ram = 4, disk = 9))

for node in nodes:
    node.add_fabnet()
slice.submit()

In [ ]:
nodes = slice.get_nodes()
nodes[0].get_ssh_command()

In [ ]:
# Install general build tools
nodes[0].execute("sudo apt-get update -y -qq")
nodes[0].execute("sudo apt-get install -y build-essential clang")

In [ ]:
# Install the Rust toolchain
nodes[0].execute("curl https://sh.rustup.rs -sSf | sh -s -- -y")

In [ ]:
# Clone rucompart
nodes[0].execute("git clone https://github.com/spaghetus/rucompart || true")

In [ ]:
# Install the rucompart db example
# ...This takes a while, too.
nodes[0].execute("""
cd ~/rucompart
git pull
~/.cargo/bin/rustup run stable cargo b --example rucompart-db --release
mkdir -p ~/.cargo/bin
ln -f target/release/examples/rucompart-db ~/.cargo/bin/rucompart-db
""")
nodes[0].download_file(
    remote_file_path = "/home/ubuntu/.cargo/bin/rucompart-db",
    local_file_path = "./rucompart-db",
)

In [ ]:
# Copy the example to the other nodes
for node in nodes[1:]:
    print(node.get_name())
    node.execute("rm -rf ~/.cargo; mkdir -p ~/.cargo/bin")
    node.upload_file(
        local_file_path = "./rucompart-db",
        remote_file_path = "/home/ubuntu/.cargo/bin/rucompart-db",
    )

In [ ]:
# Build up a table of each node's l3 IP, since finding them is pretty slow.
s_nodes = slice.get_nodes()
def ip_of(node):
    iface = node.get_interfaces()[0]
    ifname = iface.get_device_name()
    for addr in node.get_ip_addrs():
        if addr['ifname'] == ifname:
            break
    for addr in addr['addr_info']:
        if addr['family'] == 'inet':
            break
    return addr['local']
ips = {}
for node in s_nodes:
    ips[node] = ip_of(node)

In [ ]:
# For some reason, sometimes nodes fail to provision?
# This filters out broken nodes.
def node_is_reachable(node):
    (stdin, stderr) = s_nodes[0].execute("ping -c1 {ip}".format(ip = ips[node]))
    return (stdin+stderr).find("100% packet loss") == -1
nodes = list(filter(node_is_reachable, s_nodes))

print(len(s_nodes) - len(nodes), " nodes excluded.");

In [ ]:
# Trust our peers in the firewall (since channels use arbitrary ports):
s_nodes[0].execute("sudo ufw allow 8000/tcp")
for node in s_nodes:
    for other_node in s_nodes:
        node.execute_thread("sudo ufw allow from {ip}".format(ip = ips[other_node]))

In [ ]:
# Start shards on all nodes
for node in nodes:
    node.execute_thread("sudo systemctl reset-failed; chmod +x ~/.cargo/bin/rucompart-db; sudo systemd-run -u rucompart-shard.service ~/.cargo/bin/rucompart-db -v trace guest 0.0.0.0:1234")

In [ ]:
for node in nodes:
    node.execute("systemctl status rucompart-shard")

In [ ]:
# Start manager
host_cmd = "sudo systemctl reset-failed; sudo systemd-run -u rucompart-manager.service -E ROCKET_PORT=8000 -E RUST_BACKTRACE=full ~/.cargo/bin/rucompart-db -v trace host"
for node in nodes:
    host_cmd += " -s {ip}:1234".format(ip = ips[node])
print(host_cmd)
nodes[0].execute(host_cmd)

In [ ]:
from ipywidgets import interact_manual, interact, widgets
import base64
import json

def to_b64(input):
    return base64.b64encode(bytes(input, 'utf-8')).decode('utf-8')

# Fix a bug in ipywidgets where text widgets aren't correctly handled by interact_manual
def fix_text(f):
    interactive_widget = f.widget
    for widget in interactive_widget.kwargs_widgets:
        if isinstance(widget, widgets.Text):
            widget.unobserve(interactive_widget.update, names="value")

In [ ]:
# Let's try writing some data.
@interact_manual
def put(key="foo", data = '{"any": ["valid", "json"]}'):
    data = to_b64(data)
    (stdout, stderr) = nodes[0].execute("""
    echo "{json}" | base64 -d | curl -d @- http://127.0.0.1:8000/{key}
    """.format(json = data, key = key), quiet=True)
    return json.loads(stdout) if stdout != '' else {}
fix_text(put)

In [ ]:
# Notably, the order of the enumeration is nondeterministic, since it depends on which node replies first.
@interact_manual
def list():
    (stdout, stderr) = nodes[0].execute("""
    curl http://127.0.0.1:8000/
    """, quiet=True)
    return json.loads(stdout)

In [ ]:
@interact_manual
def get(key="foo"):
    (stdout, stderr) = nodes[0].execute("""
    curl http://127.0.0.1:8000/{key}
    """.format(key = key), quiet=True)
    return json.loads(stdout)
fix_text(get)

In [ ]:
# There doesn't seem to be a multiline text widget, so you'll have to hardcode
# your query script.

# Regardless, this is a very powerful tool - this instruction uses a binary-tree
# arrangement to spread the work of your query to all of the nodes, along the
# lines of something like Hadoop.

# You should write some data (ideally numbers) into the database before running this.

script = """
// This is a "rhai" script. See the language docs at https://rhai.rs/book
fn filter(key, value) {
    type_of(value) == "i64"
}

fn map(key, value) {
    value // Try using another of these expressions instead.
    // [value] // Collect the values into a list.
    // [key] // Collect the keys into a list.
    // {let v = #{}; v.set(key, value); v} // Collect the key-value pairs into a dictionary.
}

fn reduce (left, right) {
    left + right
}
"""

src = to_b64(script)
(stdout, stderr) = nodes[0].execute("""
echo "{src}" | base64 -d | curl --data-binary @- http://127.0.0.1:8000
""".format(src = src), quiet=True)
if stdout != "":
    print(json.loads(stdout))
else:
    print(stderr)

In [ ]:
# Clean up services...
for node in s_nodes:
    node.execute_thread("sudo systemctl stop rucompart-*; sudo systemctl reset-failed")

In [ ]:
# Clean up slice.
slice.delete()